## 1. Общая идея проекта

Build Eye AI состоит из трёх основных частей:

```mermaid
flowchart LR
    User[Пользователь] --> Frontend[Vue frontend]
    Frontend --> Nginx[Nginx proxy]
    Nginx --> Backend[FastAPI backend]
    Backend --> Detector[YOLO detector]
    Backend --> Matcher[План-факт matcher]
    Backend --> Parser[Excel parser]
    Detector --> OpenCV[OpenCV]
    Detector --> Model[YOLO weights]
    Backend --> Storage[Debug images / DB]
```

Система должна:

1. принять фотографию стройплощадки;
2. определить строительную технику;
3. понять, какой этап работ сейчас активен;
4. сравнить найденную технику с нормативом;
5. вернуть статус:
   - `OK`;
   - `WARNING`;
   - `CRITICAL`;
6. объяснить причину статуса на русском языке.

---

# 2. Корень репозитория

Корневая папка:

PrimeAI

Основные элементы:

- `README.md`
- `docker-compose.yml`
- `requirements.txt`
- `.env.example`
- `.dockerignore`
- `backend`
- `frontend`
- `ml`
- `data`
- `docs`
- `MSC_lct`

Корень не должен содержать бизнес-логику. Его задача:

- организовать проект;
- описать запуск;
- связать Docker-сервисы;
- хранить данные и модели;
- запускать CI.

---

# 3. Как проект создавался бы с нуля

Разумный порядок разработки:

1. Зафиксировать предметную область.
2. Создать Python backend.
3. Описать API-контракты.
4. Написать правила сопоставления.
5. Добавить CV-модуль.
6. Подключить FastAPI.
7. Создать frontend.
8. Связать frontend и backend.
9. Добавить Docker.
10. Добавить тесты и CI.
11. Подключить базу данных и реальные модели.

Важно начинать не с интерфейса и не с нейросети.

Сначала нужно определить, какие данные проходят через систему.

Например, результат анализа должен выглядеть примерно так:

```json
{
  "active_stage": "Земляные работы / Котлован",
  "detections": [
    {
      "class_name": "excavator",
      "confidence": 0.91,
      "bbox": [100, 120, 500, 600]
    }
  ],
  "status": "CRITICAL",
  "explanation": "Не хватает dump_truck и bulldozer",
  "missing_machinery": ["dump_truck", "bulldozer"],
  "unexpected_machinery": []
}
```

Этот формат является договором между backend и frontend.

---

# 4. `backend`: серверная часть

Папка `backend` отвечает за:

- HTTP API;
- обработку изображений;
- YOLO-инференс;
- правила СМР;
- анализ Excel;
- будущую работу с базой данных.

Структура:

```text
backend/
├── app/
├── tests/
├── static/
├── models/
├── requirements.txt
└── Dockerfile.backend
```

## 4.1. `requirements.txt`

Файл `requirements.txt` содержит runtime-зависимости.

Главные пакеты:

- `fastapi` — HTTP API;
- `uvicorn` — ASGI-сервер;
- `pydantic` — валидация данных;
- `pandas` — обработка таблиц;
- `openpyxl` — чтение Excel;
- `ultralytics` — YOLO;
- `opencv-python-headless` — обработка изображений;
- `python-multipart` — загрузка файлов;
- `sqlalchemy` — будущая база данных;
- `pytest` — тесты.

Также добавлены CPU-версии PyTorch:

```text
torch==2.7.1+cpu
torchvision==0.22.1+cpu
```

Это означает, что текущий Docker-образ рассчитан на запуск без CUDA.

Почему это важно:

- обычный PyTorch может скачать CUDA runtime размером в несколько гигабайт;
- Docker Desktop на обычной Windows-машине часто не настроен на GPU;
- CPU-образ проще собрать и запустить.

Для production-инференса позже можно сделать отдельный GPU-образ.

---

## 4.2. `Dockerfile.backend`

Файл `Dockerfile.backend` описывает образ backend.

Упрощённая последовательность:

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY backend/requirements.txt ./requirements.txt
RUN pip install ...
COPY backend ./backend
COPY ml ./ml
CMD ["uvicorn", "backend.app.main:app", ...]
```

Что происходит во время сборки:

1. берётся минимальный Python-образ;
2. создаётся рабочая директория `/app`;
3. копируются зависимости;
4. устанавливаются Python-пакеты;
5. копируется исходный backend;
6. копируется папка `ml`;
7. запускается Uvicorn.

Вторая важная часть Dockerfile:

```dockerfile
pip uninstall -y opencv-python
pip install opencv-python-headless
```

Она появилась из-за ошибки:

```text
ImportError: libxcb.so.1
```

Обычный `opencv-python` ожидает Linux GUI-библиотеки. В серверном контейнере GUI не нужен, поэтому используется `opencv-python-headless`.

---

# 5. `app`: Python-приложение

Папка `app` является Python-пакетом.

Она разделена на слои:

```text
app/
├── main.py
├── core/
├── api/
├── schemas/
├── services/
└── db/
```

Идея разделения:

- `main.py` — запуск приложения;
- `api` — HTTP-маршруты;
- `schemas` — форматы данных;
- `services` — бизнес-логика;
- `core` — настройки и логирование;
- `db` — база данных.

---

# 6. `core`: настройки и логирование

## 6.1. `config.py`

Файл `config.py` содержит класс `Settings`.

Он описывает:

```python
app_name
debug
database_url
yolo_weights
static_dir
```

Настройки могут приходить из:

- значений по умолчанию;
- `.env`;
- переменных окружения Docker.

Например в Compose задаётся:

```yaml
YOLO_WEIGHTS: /app/ml/weights/best_yolo11x_construction.pt
```

Это значение попадает в окружение backend-контейнера.

Функция `get_settings()` использует `lru_cache`, чтобы настройки создавались один раз.

### Важное ограничение

Сейчас `config.py` существует, но `MachineryDetector` напрямую читает `os.getenv("YOLO_WEIGHTS")`.

То есть единая система настроек пока не доведена до конца.

Лучше было бы, чтобы detector получал настройки через `get_settings()`.

---

## 6.2. `logging.py`

Файл `logging.py` задаёт единый формат логов:

```text
время | уровень | модуль | сообщение
```

Он вызывается при старте FastAPI.

Логирование нужно для:

- времени инференса;
- ошибок модели;
- ошибок чтения изображений;
- диагностики контейнера;
- анализа инцидентов.

---

# 7. `schemas`: контракты данных

Файл `contracts.py` является фундаментом системы.

До написания detector и frontend нужно создать именно его.

## 7.1. `MachineryType`

Enum разрешённых классов:

```text
excavator
dump_truck
bulldozer
concrete_mixer
mobile_crane
crane_manipulator
roller
truck
```

Почему Enum полезен:

- ограничивает возможные значения;
- защищает API от произвольных строк;
- помогает избежать разных написаний одного класса;
- автоматически документируется в Swagger.

---

## 7.2. `DetectionItem`

Описывает один найденный объект:

```python
class_name
confidence
bbox
```

Например:

```json
{
  "class_name": "excavator",
  "confidence": 0.87,
  "bbox": [100, 200, 500, 650]
}
```

`bbox` — координаты прямоугольника:

```text
[left, top, right, bottom]
```

---

## 7.3. `StageRequirement`

Описывает норматив для этапа:

```python
stage_name
required_machinery
optional_machinery
```

Например:

```json
{
  "stage_name": "Земляные работы / Котлован",
  "required_machinery": {
    "excavator": 1,
    "dump_truck": 1,
    "bulldozer": 1
  }
}
```

---

## 7.4. `IncidentStatus`

Возможные результаты:

```text
OK
WARNING
CRITICAL
```

---

## 7.5. `AnalyzeResponse`

Это главный ответ API.

Он объединяет:

- время анализа;
- текущий этап;
- детекции;
- итоговый статус;
- объяснение;
- отсутствующую технику;
- неожиданную технику.

Frontend ориентируется на этот контракт.

---

# 8. `services/ontology.py`: база знаний СМР

Файл `ontology.py` не использует нейросеть.

Он содержит нормативные правила.

Например:

```text
Подготовка территории / Демонтаж
    excavator
    dump_truck

Земляные работы / Котлован
    excavator
    dump_truck
    bulldozer

Устройство фундамента / Сваи
    mobile_crane
    concrete_mixer
    truck
```

Это не машинное обучение. Это обычная экспертная логика.

## `get_stage_rules()`

Функция принимает пользовательский текст:

```text
"котлован"
```

и пытается найти подходящий норматив.

Поддерживаются:

- точное совпадение;
- поиск по ключевым словам;
- нормализация регистра;
- замена `ё` на `е`.

Например:

```text
"Работы в котловане"
```

может быть сопоставлено с:

```text
"Земляные работы / Котлован"
```

Если этап неизвестен, возвращается пустое правило.

Это лучше, чем молча выбрать неправильный этап.

---

# 9. `services/matcher.py`: план-факт анализ

Файл `matcher.py` отвечает за решение:

> соответствует ли обнаруженная техника нормативу этапа?

Основная функция:

```python
evaluate_compliance(stage_name, detected_classes)
```

Например:

```python
evaluate_compliance(
    "котлован",
    ["excavator"]
)
```

Внутри происходит следующее.

## Шаг 1. Поиск правил

Matcher вызывает:

```python
get_stage_rules(stage_name)
```

Получает список обязательной техники.

## Шаг 2. Подсчёт фактической техники

Используется `Counter`.

Например:

```text
excavator: 1
dump_truck: 0
bulldozer: 0
```

## Шаг 3. Поиск дефицита

Если обязательная техника отсутствует, добавляется в `missing_machinery`.

## Шаг 4. Поиск нетипичной техники

Если техника обнаружена, но не входит в required или optional, она попадает в `unexpected_machinery`.

## Шаг 5. Выбор статуса

Правила:

```text
Есть missing -> CRITICAL
Нет missing, но есть unexpected -> WARNING
Неизвестный этап -> WARNING
Всё соответствует -> OK
```

## Шаг 6. Explainable AI

Matcher формирует текст на русском языке.

Это не генеративная нейросеть. Это шаблонное объяснение на основании конкретных данных.

Например:

```text
На этапе «Земляные работы / Котлован» не хватает обязательной техники:
dump_truck, bulldozer.
Это критичное отклонение плана-факта...
```

Плюс такого подхода — объяснение детерминированное и проверяемое.

---

# 10. `services/detector.py`: компьютерное зрение

Файл `detector.py` отвечает за YOLO.

Класс:

```python
MachineryDetector
```

## Создание detector

При создании:

1. читается путь к весам;
2. проверяется наличие `.pt`;
3. если веса есть, загружается Ultralytics YOLO;
4. если веса отсутствуют, модель остаётся `None`.

## Метод `detect()`

Он принимает:

```python
image_bytes: bytes
```

То есть файл из HTTP-загрузки.

Затем:

1. bytes превращаются в NumPy-массив;
2. OpenCV декодирует изображение;
3. YOLO выполняет inference;
4. результат преобразуется в `DetectionItem`;
5. неизвестные классы отбрасываются.

## Нормализация имён

Модель может вернуть:

```text
cement mixer
cement-mixer
mixer
```

Detector приводит варианты к единому виду:

```text
concrete_mixer
```

Также поддерживаются aliases:

```text
crane -> mobile_crane
lorry -> truck
tipper -> dump_truck
digger -> excavator
road_roller -> roller
```

## Fallback

Если настоящих весов нет:

- изображение всё равно проверяется;
- создаётся синтетический `excavator`;
- confidence равен `0.01`;
- bbox занимает центральную часть изображения.

Это нужно, чтобы приложение можно было протестировать до обучения модели.

Но важно понимать:

> fallback не является настоящей детекцией.

Он нужен только для smoke-теста интерфейса и API.

---

# 11. `api/v1`: HTTP-маршруты

Папка `v1` переводит Python-функции в HTTP endpoints.

## 11.1. `router.py`

Файл `router.py` создаёт общий префикс:

```text
/api/v1
```

И подключает:

- `monitoring`;
- `schedule`;
- `analytics`.

Версионирование нужно, чтобы в будущем можно было создать:

```text
/api/v2
```

не ломая старый frontend.

---

## 11.2. `monitoring.py`

Файл `monitoring.py` содержит основной рабочий поток.

### `GET /api/v1/health`

Проверяет, жив ли backend.

Ответ:

```json
{
  "status": "ok"
}
```

### `POST /api/v1/analyze`

Принимает:

- `image`;
- `stage_name`.

Поток:

```text
HTTP file
  -> image bytes
  -> detector.detect()
  -> matcher.evaluate_compliance()
  -> draw bbox
  -> AnalyzeResponse
```

### Сохранение debug-изображения

После анализа OpenCV:

- открывает исходный снимок;
- рисует bbox;
- пишет название класса;
- пишет confidence;
- сохраняет `.jpg`.

Путь:

```text
backend/static/debug/
```

В Docker этот каталог подключён через volume:

```yaml
backend_static:/app/backend/static
```

### `POST /api/v1/batch-analyze`

Принимает несколько файлов:

```text
images[]
stage_name
```

Каждый файл обрабатывается отдельно, но этап один и тот же.

Сейчас результаты возвращаются только в HTTP-ответе и не сохраняются в БД.

---

## 11.3. `schedule.py`

Файл `schedule.py` отвечает за Excel.

Endpoint:

```text
POST /api/v1/schedule/upload
```

Он:

1. принимает файл;
2. читает bytes;
3. вызывает `parse_schedule()`;
4. возвращает строки таблицы.

---

## 11.4. `analytics.py`

Файл `analytics.py` пока содержит заглушку:

```json
{
  "total_incidents": 0,
  "by_status": {
    "OK": 0,
    "WARNING": 0,
    "CRITICAL": 0
  }
}
```

Он не читает реальные результаты анализа.

Чтобы analytics стал настоящим, нужно:

1. сохранять каждый AnalyzeResponse;
2. записывать IncidentAlert;
3. считать записи по статусам;
4. фильтровать по дате и этапу.

---

# 12. `main.py`: точка запуска backend

Файл `main.py` создаёт объект:

```python
app = FastAPI(...)
```

Он:

1. создаёт FastAPI;
2. включает логирование;
3. настраивает CORS;
4. подключает `/api/v1`.

CORS разрешает:

```text
http://localhost:5173
http://localhost:3000
```

Это нужно, потому что frontend и backend работают на разных портах.

Запуск внутри Docker:

```text
uvicorn backend.app.main:app --host 0.0.0.0 --port 8000
```

`0.0.0.0` важен: без него сервер может быть доступен только внутри контейнера.

---

# 13. `db`: база данных

Папка `db` подготовлена для SQLAlchemy.

## `session.py`

Файл `session.py`:

- создаёт engine;
- создаёт `SessionLocal`;
- описывает базовый класс ORM;
- предоставляет `get_db()`.

## `models.py`

Файл `models.py` описывает три сущности.

### `Stage`

Этап календарного графика.

### `MachineryDetection`

Сохранённая детекция:

- класс;
- confidence;
- bbox;
- этап;
- время.

### `IncidentAlert`

Инцидент:

- статус;
- объяснение;
- этап;
- время.

## Текущее состояние БД

База пока не включена в рабочий поток.

Пока отсутствуют:

- `Base.metadata.create_all()`;
- Alembic;
- `Depends(get_db)`;
- запись детекций;
- запись инцидентов;
- чтение аналитики из SQL.

То есть модели существуют, но текущий API работает в основном в памяти.

---

# 14. `frontend`: пользовательский интерфейс

Папка `frontend` содержит Vue-приложение.

Основные файлы:

```text
frontend/
├── package.json
├── index.html
├── vite.config.js
├── nginx.conf
├── Dockerfile.frontend
└── src/
```

## `package.json`

Файл `package.json`:

- `vue` — UI-фреймворк;
- `axios` — HTTP-запросы;
- `vite` — сборка;
- `@vitejs/plugin-vue` — поддержка `.vue`.

Команда:

```json
"build": "vite build"
```

---

## `vite.config.js`

Файл `vite.config.js` подключает Vue-плагин:

```javascript
plugins: [vue()]
```

Скобки `vue()` обязательны.

Если написать:

```javascript
plugins: [vue]
```

Vite не будет правильно обрабатывать `.vue`-файлы.

---

## `main.js`

Файл `main.js` монтирует Vue-приложение в HTML-элемент:

```html
<div id="app"></div>
```

Именно с него начинается frontend runtime.

---

## `App.vue`

Файл `App.vue` выбирает главный экран:

```text
App.vue
  -> DashboardView.vue
```

Сейчас нет Vue Router, поэтому приложение сразу показывает dashboard.

---

# 15. Основной frontend-поток анализа

## `DashboardView.vue`

Файл `DashboardView.vue` связывает:

- Navbar;
- UploadModal;
- IncidentCard.

Он хранит состояние:

```javascript
const analysis = ref(null)
```

После успешного API-запроса:

```text
UploadModal emits analyzed
DashboardView updates analysis
IncidentCard displays result
```

---

## `UploadModal.vue`

Файл `UploadModal.vue` выполняет главный пользовательский сценарий.

Он содержит:

- select этапа;
- input для изображения;
- кнопку анализа;
- состояние загрузки;
- сообщение об ошибке.

При отправке:

```javascript
const formData = new FormData()
formData.append('image', selectedFile)
formData.append('stage_name', stageName)
api.post('/analyze', formData)
```

После ответа:

```javascript
emit('analyzed', data)
```

Это событие получает `DashboardView`.

---

## `api.js`

Файл `api.js` создаёт Axios-клиент:

```javascript
baseURL: '/api/v1'
```

Почему относительный адрес важен:

- браузер обращается к `localhost:5173`;
- Nginx получает запрос `/api/v1`;
- Nginx проксирует его в контейнер `backend:8000`.

Frontend не обязан знать Docker-имя backend.

---

## `nginx.conf`

Файл `nginx.conf` выполняет две задачи.

### Статические файлы

```text
/
```

отдаёт собранный Vue-сайт.

### API proxy

```text
/api/
```

перенаправляется сюда:

```text
http://backend:8000/api/
```

Также установлен лимит:

```nginx
client_max_body_size 20M;
```

Без него Nginx возвращал:

```text
413 Request Entity Too Large
```

---

## `IncidentCard.vue`

Файл `IncidentCard.vue` показывает:

- статус;
- объяснение;
- отсутствующую технику;
- нетипичную технику.

До подключения API карточка была статической. Теперь она получает данные через props.

---

# 16. Пока незавершённые frontend-компоненты

## `Navbar.vue`

Пока отображает только название и статический текст.

Не подключён к:

- healthcheck;
- текущему статусу стройки;
- авторизации;
- навигации.

## `PhotoViewer.vue`

Заготовка для:

- показа исходного изображения;
- показа bbox;
- отображения labels;
- zoom.

Пока основной экран его не использует.

## `GanttChart.vue`

Пока placeholder.

Для настоящего графика нужны:

- даты начала и окончания;
- текущий процент выполнения;
- активный этап;
- связь с Excel-расписанием;
- план и факт.

## `TimelineView.vue`

Будущий детальный таймлайн СМР.

## `CameraSettingsView.vue`

Будущая страница рекомендаций по камерам.

---

# 17. Docker Compose

Файл `docker-compose.yml` запускает два сервиса.

## Backend

```yaml
backend:
  ports:
    - "8000:8000"
```

Снаружи backend доступен на:

```text
http://localhost:8000
```

## Frontend

```yaml
frontend:
  ports:
    - "5173:80"
```

Снаружи frontend доступен на:

```text
http://localhost:5173
```

Внутри контейнера Nginx слушает порт `80`, но наружу он опубликован как `5173`.

## `depends_on`

Frontend ждёт:

```yaml
backend:
  condition: service_healthy
```

Backend должен сначала пройти:

```text
GET /api/v1/health
```

## Docker volume

```yaml
backend_static:/app/backend/static
```

Volume нужен, чтобы debug-изображения не исчезали при пересоздании контейнера.

---

# 18. Полный запрос пользователя

Когда пользователь нажимает кнопку анализа:

```text
1. Browser opens frontend on localhost:5173
2. Vue displays DashboardView
3. User selects image and stage
4. UploadModal creates FormData
5. Axios sends POST /api/v1/analyze
6. Nginx receives /api/v1/analyze
7. Nginx forwards request to backend:8000
8. FastAPI receives UploadFile
9. OpenCV decodes image
10. YOLO detects machinery
11. Matcher loads stage rules
12. Matcher calculates missing/unexpected classes
13. Backend creates explanation
14. OpenCV saves debug image
15. FastAPI returns AnalyzeResponse
16. Vue stores response in analysis
17. IncidentCard renders status
```

Это главный сквозной сценарий репозитория.

---

# 19. ML-часть

Папка `ml` нужна для экспериментов, обучения и проверки моделей.

В ней есть:

- `train.py`;
- `download_model.py`;
- `test_inference.py`;
- `datasets/download_and_merge.py`;
- `notebooks/`;
- `weights/`;
- `output/`.

## `download_model.py`

Должен загружать или подготавливать модель.

## `train.py`

Предназначен для fine-tuning.

Текущая реализация ещё не является полноценным pipeline обучения.

## `download_and_merge.py`

Предполагается для:

- загрузки датасетов;
- приведения классов;
- объединения источников;
- подготовки YOLO-структуры.

## `test_inference.py`

Используется для ручной проверки модели на фотографиях.

Важно: некоторые ML-скрипты используют жёсткий Windows-путь. В будущем лучше использовать:

```python
Path(__file__).resolve().parents[1]
```

чтобы скрипт работал и в Docker, и на другой машине.

---

# 20. `data`

Папка `data` содержит исходные данные.

Сейчас там:

```text
data/
├── raw_photos/
└── Сводный перечень строительных работ_ЛТЦ.xlsx
```

## `raw_photos`

Фотографии строительной площадки.

Они нужны для:

- ручного smoke-теста;
- оценки fallback;
- тестирования модели;
- демонстрации интерфейса.

## Excel-файл

Excel содержит исходный перечень строительных работ.

Сейчас backend читает Excel обобщённо:

```python
pandas.read_excel(...)
```

Он возвращает строки таблицы, но пока не превращает их автоматически в:

```text
StageRequirement
```

Это следующий важный этап развития проекта.

---

# 21. `docs`

Папка `docs` содержит документацию.

Сейчас там:

`Camera_Placement_Guide.md`

Там должны находиться:

- правила установки камер;
- ограничения углов обзора;
- рекомендации по освещению;
- допустимая высота и дистанция;
- зоны перекрытия;
- рекомендации для YOLO.

---

# 22. Тесты

Папка `tests` содержит тесты backend.

## `test_matcher.py`

Проверяет предметную область:

- дефицит техники даёт `CRITICAL`;
- полный комплект даёт `OK`;
- лишняя техника даёт `WARNING`.

Это самые ценные тесты на текущем этапе, потому что matcher является ядром бизнес-логики.

## `test_api.py`

Проверяет:

```text
GET /api/v1/health
```

Пока не проверяются:

- `/analyze`;
- `/batch-analyze`;
- `/schedule/upload`;
- `/analytics/summary`;
- ошибки загрузки;
- реальные изображения.

---

# 23. CI

Файл `ci.yml` запускается в GitHub Actions.

Сейчас он проверяет:

```text
ruff check backend
black --check backend
```

То есть CI проверяет стиль и базовые ошибки, но пока не запускает:

- pytest;
- frontend build;
- Docker build;
- проверку Vue;
- inference;
- Excel parser.

Логичный следующий шаг:

```yaml
- run: pytest backend/tests
- run: docker compose build
```

---

# 24. Что сейчас реально работает

Рабочий путь:

1. Docker Compose запускает backend.
2. Backend проходит healthcheck.
3. Frontend собирается Vite.
4. Nginx отдаёт Vue.
5. Форма принимает изображение.
6. Изображение отправляется в `/api/v1/analyze`.
7. Backend выполняет detector.
8. При отсутствии весов используется fallback.
9. Matcher возвращает статус.
10. Frontend отображает результат.

Проверенные адреса:

```text
http://localhost:5173
http://localhost:8000/docs
http://localhost:8000/api/v1/health
```

---

# 25. Что пока является демонстрацией

Следующие части пока не production-ready:

- fallback-детекция;
- настоящие YOLO-веса;
- сохранение анализа в БД;
- аналитика;
- Excel-to-ontology автоматическая связь;
- GanttChart;
- PhotoViewer;
- TimelineView;
- batch UI;
- авторизация;
- мониторинг;
- GPU-инференс;
- миграции базы данных.

Главное различие:

> Сейчас есть рабочий вертикальный срез, но не вся задуманная система.

Вертикальный срез — это минимальный путь:

```text
фото -> детекция -> правила -> статус -> экран
```

Это правильный первый этап разработки.

---

# 26. Как самому строить такой проект

Рекомендуемая последовательность:

## Этап 1. Доменная модель

Создать:

- `MachineryType`;
- `IncidentStatus`;
- `DetectionItem`;
- `StageRequirement`;
- `AnalyzeResponse`.

## Этап 2. Детерминированные правила

Реализовать matcher без нейросети.

Проверить тестами:

```text
missing -> CRITICAL
complete -> OK
unexpected -> WARNING
```

## Этап 3. API

Создать endpoint, который временно принимает список классов напрямую.

Проверить Swagger.

## Этап 4. Detector

Подключить YOLO и преобразовать его результат к `DetectionItem`.

Добавить fallback.

## Этап 5. Изображения

Добавить OpenCV:

- чтение;
- bbox;
- сохранение debug-файла.

## Этап 6. Frontend

Сначала сделать простую форму:

- выбрать изображение;
- выбрать этап;
- отправить запрос;
- показать JSON.

Только потом добавлять красивый dashboard.

## Этап 7. Docker

Сначала отдельно собрать backend:

```powershell
docker compose build backend
```

Потом отдельно frontend:

```powershell
docker compose build frontend
```

Потом запустить всё:

```powershell
docker compose up -d
```

## Этап 8. База данных

Подключить persistence:

- создать таблицы;
- сохранять анализы;
- сохранять инциденты;
- подключить analytics.

## Этап 9. Реальное расписание

Научить Excel parser преобразовывать таблицу в этапы и нормативы.

## Этап 10. Production ML

Добавить:

- реальные дообученные веса;
- confidence threshold;
- валидацию модели;
- mAP;
- GPU;
- batching;
- profiling;
- версионирование моделей.

---

# 27. Главная архитектурная мысль

Репозиторий разделён не случайно:

```text
schemas   = какие данные разрешены
ontology  = какие правила предметной области
detector  = что увидела модель
matcher   = насколько это соответствует плану
api       = как получить результат по HTTP
frontend  = как показать результат человеку
docker    = как всё запустить одинаково
tests     = как проверить, что правила не сломались
ml        = как развивать модель
data      = на чём модель и parser работают
docs      = как объяснить систему пользователю
```

Самая важная цепочка:

```text
contracts
  -> ontology
  -> matcher
  -> detector
  -> monitoring API
  -> Vue frontend
  -> Docker
```

Если строить проект именно в таком порядке, каждый следующий слой получает стабильный интерфейс от предыдущего и не приходится переписывать всю систему целиком.